## 域適應（DaNN）：解決跨機台/跨批次數據分佈波動

- 目標：深入理解半導體場景中「領域偏移（Domain Shift）」的物理成因。掌握領域對抗神經網路（Domain Adversarial Neural Network, DaNN）的核心架構。利用 PyTorch 實作包含特徵提取器（Feature Extractor）、標籤預測器（Label Predictor）與領域分類器（Domain Classifier）的對抗訓練 Pipeline，使模型能完美克服跨機台與跨批次的資料分佈波動。


### 1. 為什麼半導體場景強烈需要「域適應 (Domain Adaptation)」？

- 核心：
    - 當我們在 A 機台（源領域，Source Domain）收集了大量標註好的晶圓缺陷圖並訓練好一個 CNN 分類器，直接拿到 B 機台（目標領域，Target Domain）使用時，精準度往往會暴跌。
    - 物理成因：不同測試機台（如愛德萬 vs 泰瑞達）或不同生產批次之間，由於硬體探針磨損、測試電流雜訊（Noise Floor）、或是腔室真空度的微小差異，會導致晶圓圖的背景噪點、對比度與缺陷特徵的分佈產生漂移。
    - 傳統做法的困境：對每個新機台重新人工標註幾萬片晶圓是不可能的，成本太高。
    - DaNN 的解決之道：利用對抗訓練，迫使特徵提取器「只提取與缺陷本質相關的特徵，同時主動遺忘、濾除與機台廠牌相關的特徵」。從而實現無監督遷移學習（目標領域不需標籤）。


### 2. DaNN 核心架構：反向梯度層 (GRL) 與三大模組設計

- DaNN 的精髓在於 梯度反向層 (Gradient Reversal Layer, GRL)。在正向傳播時，它只是一個恆等映射；但在反向傳播時，它會把傳過來的梯度乘以一個負數，以此來對抗、干擾領域分類器，讓特徵提取器學到「跨機台通用（Domain-Invariant）」的強健特徵。


In [ ]:
import torch
import torch.nn as nn
from torch.autograd import Function


# --- 實作梯度反向層 (Gradient Reversal Layer) ---
class GradientReversalFn(Function):
    @staticmethod
    def forward(ctx, x, alpha):
        ctx.alpha = alpha
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        # 反向傳播時，將梯度乘以 -alpha，實現對抗
        output = grad_output.neg() * ctx.alpha
        return output, None


def grad_reverse(x, alpha=1.0):
    return GradientReversalFn.apply(x, alpha)


# --- 定義 DaNN 三大核心模組 ---
class WaferFeatureExtractor(nn.Module):
    """特徵提取器：負責從晶圓圖中提取深層特徵"""

    def __init__(self):
        super(WaferFeatureExtractor, self).__init__()
        self.feature = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # [B, 1, 28, 28] -> [B, 16, 14, 14]
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # [B, 16, 14, 14] -> [B, 32, 7, 7]
            nn.Flatten(),  # 32 * 7 * 7 = 1568
        )

    def forward(self, x):
        return self.feature(x)


class WaferLabelPredictor(nn.Module):
    """標籤預測器：負責分類缺陷類型 (Scratch, Ring, Cluster)"""

    def __init__(self, num_classes=3):
        super(WaferLabelPredictor, self).__init__()
        self.class_classifier = nn.Sequential(
            nn.Linear(1568, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.class_classifier(x)


class WaferDomainClassifier(nn.Module):
    """領域分類器：負責分辨這片晶圓來自哪台機台 (例如 0:機台A, 1:機台B)"""

    def __init__(self):
        super(WaferDomainClassifier, self).__init__()
        self.domain_classifier = nn.Sequential(
            nn.Linear(1568, 64),
            nn.ReLU(),
            nn.Linear(64, 2),  # 二分類：機台 A 或 機台 B
        )

    def forward(self, x, alpha):
        # 關鍵點：在輸入領域分類器前，先經過梯度反向層
        reversed_x = grad_reverse(x, alpha)
        return self.domain_classifier(reversed_x)

### 3. 對抗訓練流水線 (train_domain_adaptation) 模擬

- 實作：我們模擬同時輸入「有機台標籤與缺陷標籤的機台 A（Source）」以及「有機台標籤但完全沒有缺陷標籤的機台 B（Target）」進行聯合對抗訓練。


In [ ]:
import torch.optim as optim


def train_domain_adaptation():
    """對應專案：train_domain_adaptation() 核心架構演示"""
    # 初始化模型與優化器
    fe = WaferFeatureExtractor()
    lp = WaferLabelPredictor(num_classes=3)
    dc = WaferDomainClassifier()

    # 聯合優化所有子模型的參數
    optimizer = optim.Adam(
        list(fe.parameters()) + list(lp.parameters()) + list(dc.parameters()), lr=0.001
    )

    # 損失函數
    criterion_class = nn.CrossEntropyLoss()
    criterion_domain = nn.CrossEntropyLoss()

    # 模擬一組 Batch 的產線輸入數據 (影像大小 28x28)
    batch_size = 4
    # 源領域 A：包含晶圓圖、缺陷標籤 (0,1,2)、領域標籤全部為 0 (代表機台 A)
    src_img = torch.randn(batch_size, 1, 28, 28)
    src_class_label = torch.tensor([0, 1, 2, 1], dtype=torch.long)
    src_domain_label = torch.zeros(batch_size, dtype=torch.long)

    # 目標領域 B：包含晶圓圖、領域標籤全部為 1 (代表機台 B) -> 【完全沒有缺陷標籤】
    tgt_img = torch.randn(batch_size, 1, 28, 28)
    tgt_domain_label = torch.ones(batch_size, dtype=torch.long)

    print(">>> 啟動 DaNN 步驟化前向與反向傳播測試...")

    # 前向傳播與損失計算
    alpha = 0.5  # 梯度反向權重

    # 處理源領域 A
    src_feat = fe(src_img)
    src_class_pred = lp(src_feat)
    src_domain_pred = dc(src_feat, alpha)

    loss_src_class = criterion_class(src_class_pred, src_class_label)
    loss_src_domain = criterion_domain(src_domain_pred, src_domain_label)

    # 處理目標領域 B
    tgt_feat = fe(tgt_img)
    tgt_domain_pred = dc(tgt_feat, alpha)
    loss_tgt_domain = criterion_domain(tgt_domain_pred, tgt_domain_label)

    # 總損失 = 缺陷分類損失 + 領域對抗損失 (兩機台加總)
    total_loss = loss_src_class + (loss_src_domain + loss_tgt_domain)

    # 反向傳播更新
    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    print("=" * 60)
    print(f"聯合對抗訓練反向傳播成功！")
    print(f"任務分類損失 (Class Loss): {loss_src_class.item():.4f}")
    print(
        f"領域對抗損失 (Domain Loss): {(loss_src_domain + loss_tgt_domain).item():.4f}"
    )
    print("=" * 60)


# 執行測試
train_domain_adaptation()

- 總結：半導體測試落地最常遇到的骨頭（硬骨頭），就是神經網路在面對跨機台、跨批次的資料分佈波動（Domain Shift）時，分類精準度會大幅滑落。這在物理上是由於硬體探針磨損、測試環境底噪不一導致的。為了解決這個問題，我沒有採用成本高昂的重新標註，而是引入了域適應（Domain Adaptation）技術，在專案中實作了 DaNN（領域對抗神經網路）。我將網路拆解為 Feature Extractor、Label Predictor 與 Domain Classifier 三個子模組，並在中間插入自訂的 梯度反向層 (Gradient Reversal Layer)。在訓練過程中，模型利用已知標籤的 A 機台數據去學習缺陷分類，同時利用完全無標籤的 B 機台數據與 A 機台數據一起投餵給領域分類器。透過反向梯度對抗，我們強迫特徵提取器抹除掉所有與『特定機台硬體指紋』相關的特徵，只留下最純粹、跨機台通用的缺陷拓撲特徵。這讓我們的演算法不用經過二次標註，就能流暢地遷移部署到全新產線的邊緣設備上。
